# Time Series Course — m10-f3 (block 3)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
# Experiment: window size vs accuracy
# Run this for lookback in [7, 14, 30, 60, 90, 180]

results = {}
for lookback in [7, 14, 30, 60, 90, 180]:
    X_tr, y_tr = create_windows(train_norm, lookback)
    X_te, y_te = create_windows(test_norm, lookback)

    X_tr_t = torch.FloatTensor(X_tr).unsqueeze(-1)
    y_tr_t = torch.FloatTensor(y_tr).unsqueeze(-1)
    X_te_t = torch.FloatTensor(X_te).unsqueeze(-1)

    model = GRUForecaster(input_size=1, hidden_size=64, num_layers=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    # Quick train (20 epochs)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True
    )
    for epoch in range(20):
        model.train()
        for xb, yb in loader:
            pred = model(xb); loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        pred = model(X_te_t).numpy().flatten()
    mae = np.abs(pred * train_std + train_mean - (y_te * train_std + train_mean)).mean()
    results[lookback] = mae
    print(f"Lookback={lookback:>3d}: MAE={mae:.2f}°C")

# Typically: MAE drops from L=7 to L=60, then plateaus or rises